# Top-10 HOR lengths per chromosome (centroAnno)

Per-chromosome count distribution of the ten most common higher-order repeat (HOR)
lengths, from centroAnno HOR-decompose results; bars coloured by average identity.


In [ ]:
# Project root — edit for your environment.
PROJ_ROOT = "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj"


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.axes_grid1 import make_axes_locatable


In [ ]:
chr_colors = {
    'chr1': '#FF6B6B', 'chr2': '#4ECDC4', 'chr3': '#45B7D1', 'chr4': '#FFA07A',
    'chr5': '#92D050', 'chr6': '#D35FB7', 'chr7': '#FFC000', 'chr8': '#00B0F0',
    'chr9': '#A2D96C', 'chr10': '#C00000', 'chr11': '#7030A0', 'chr12': '#FF5733',
    'chr13': '#00AEEF', 'chr14': '#FF99CC', 'chr15': '#8FD8D8', 'chr16': '#F6546A',
    'chr17': '#468499', 'chr18': '#FFD700', 'chr19': '#088DA5', 'chr20': '#F08080',
    'chr21': '#6A5ACD', 'chr22': '#65B891', 'chr23': '#FFA500', 'chr24': '#BA55D3',
    'chr25': '#9370DB', 'chr26': '#3CB371', 'chr27': '#7B68EE', 'chr28': '#40E0D0',
    'chrX': '#FF1493',  # Distinct pink for X
    'chrY': '#14FF82'   # Dark blue for Y
}

In [ ]:

# Create a figure with subplots
fig, axes = plt.subplots(6, 5, figsize=(20, 24))
axes = axes.flatten()

# Find global min and max for identity scores across all chromosomes
all_identities = []

for chrom in chr_colors.keys():
    try:
        hor_char=pd.read_csv(f"{PROJ_ROOT}/output/outputs-from-centraAnno/hifiasm-0414/""+chrom+"_horDecomposedResult.csv",names=["chr","HOR_template","start","end","repblock_ident","HOR_size","score"],sep=",")

        hor_df = hor_char["HOR_size"].value_counts().rename_axis('hor_len').reset_index(name='counts')
        top_10 = hor_df.head(10)
        
        for mono_len in top_10['hor_len']:
            avg_ident = hor_char[hor_char['HOR_size'] == mono_len]['repblock_ident'].mean()
            all_identities.append(avg_ident)
    except:
        continue

# Create global colormap
if all_identities:
    global_min = min(all_identities)
    global_max = max(all_identities)
    cmap = plt.cm.viridis
    norm = plt.Normalize(global_min, global_max)

# Loop through chromosomes
for i, chrom in enumerate(chr_colors.keys()):
    if i >= 30:
        break
        
    try:
        hor_char=pd.read_csv(f"{PROJ_ROOT}/output/outputs-from-centraAnno/hifiasm-0414/""+chrom+"_horDecomposedResult.csv",names=["chr","HOR_template","start","end","repblock_ident","HOR_size","score"],sep=",")
        hor_df = hor_char["HOR_size"].value_counts().rename_axis('hor_len').reset_index(name='counts')
        top_10 = hor_df.head(10)
        
        avg_identities = []
        for mono_len in top_10['hor_len']:
            avg_ident = hor_char[hor_char['HOR_size'] == mono_len]['repblock_ident'].mean()
            avg_identities.append(avg_ident)
        
        top_10['avg_identity'] = avg_identities
        colors = [cmap(norm(ident)) for ident in top_10['avg_identity']]
        
        ax = axes[i]
        bars = ax.bar(range(len(top_10)), top_10['counts'], color=colors, edgecolor='black')
        
        ax.set_xticks(range(len(top_10)))
        ax.set_xticklabels(top_10['hor_len'], rotation=45,size=13)
        ax.tick_params(axis='y', labelsize=13)
        ax.set_xlabel('HOR Length',size=12)
        ax.set_ylabel('Count',size=12)
        ax.set_title(f'{chrom}',size=12)
        
        # Add individual colorbar (smaller)
                divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        plt.colorbar(sm, cax=cax)
        
    except Exception as e:
        print(f"Error with {chrom}: {e}")
        axes[i].set_visible(False)

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()